# Agentic Workflow Patterns — A Medical AI Case Study

**Model:** `gpt-5-nano` via the OpenAI Responses API
**Domain:** Clinical decision support (educational case study)
**What you'll learn:** how to design *reliable* LLM systems using structured workflows,
tool use, feedback loops, and multi-step orchestration patterns.

---

# Choosing the Right LLM Approach

## Scenario 1 — Polishing a message

Your manager sends a rough note:

> "Tell the team the client meeting moved to Friday and we need the deck by tomorrow."

You want the system to rewrite it into a polished, professional email.

**Question:** Which approach would you choose?

---

## Scenario 2 — Turning a lecture into study material

You have a raw transcript from a 45-minute class. You want the system to:

1. clean the transcript,
2. extract the key topics,
3. summarize each topic,
4. generate 5 quiz questions.

The sequence is already known in advance.

**Question:** Which approach would you choose?

---

## Scenario 3 — Handling support tickets

A company receives customer emails. The system must:

* classify whether the issue is billing, technical, or account-related,
* search the right knowledge base,
* draft a reply,
* escalate to a human if confidence is low.

The company wants a predictable outer structure, but some flexibility inside it.

**Question:** Which approach would you choose?

---

## Scenario 4 — Travel planning with changing constraints

A user says:

> "Find me the best flight from Delhi to Bangalore for tomorrow morning, under ₹8,000, with arrival before 10 AM, and avoid long layovers."

The system may need to search multiple times, compare options, inspect details, and revise its strategy if nothing matches.

**Question:** Which approach would you choose?

---

## Scenario 5 — Resume tailoring

A job seeker uploads a resume and a job description. The system should:

* extract relevant skills from the job description,
* compare them against the resume,
* rewrite the resume summary,
* draft a short cover letter.

This process is repeated for many job applications using the same predictable steps.

**Question:** Which approach would you choose?

---

## Scenario 6 — Investigating a failed cloud deployment

A developer says:

> "Yesterday's deployment failed. Figure out what went wrong."

The system may need to:

* inspect CI/CD logs,
* check recent code changes,
* compare with the last successful deployment,
* inspect config files,
* possibly run tests,
* keep investigating until it finds the likely root cause.

The path is not known beforehand.

**Question:** Which approach would you choose?

---

## Scenario 7 — Expense receipt processing

A finance team wants to automate reimbursement. The system must:

* OCR a receipt,
* extract merchant, date, and amount,
* classify the expense category,
* check policy rules,
* flag suspicious or incomplete cases for review.

The company wants a strong predefined process, but the model may need to make judgments at certain steps.

**Question:** Which approach would you choose?

---

## Scenario 8 — Summarizing one legal clause

A user pastes one paragraph from a contract and asks:

> "Explain this clause in simple language."

No tool use is required. No multi-step flow is required. The user just wants a direct explanation.

**Question:** Which approach would you choose?

---

## Scenario 9 — Research assistant for buying a laptop

A user says:

> "Find me the best laptop under my budget for video editing, compare current options, and explain the tradeoffs."

The system may need to:

* search current products,
* compare specs,
* filter by budget,
* inspect reviews,
* revise recommendations if constraints conflict.

There is uncertainty, and the number of steps depends on what the system finds.

**Question:** Which approach would you choose?

---

## Scenario 10 — Weekly business report generation

Every Friday, a company wants a system to:

1. read sales data,
2. summarize trends,
3. highlight anomalies,
4. generate a manager-friendly report.

The structure is stable every week, and the business wants consistency and easy debugging.

**Question:** Which approach would you choose?


## Answers key

### 1. Polishing a message

**Best fit:** Single-shot
Reason: one prompt, one rewrite, no tools, no iteration.

### 2. Turning a lecture into study material

**Best fit:** Prompt chain
Reason: fixed multi-step process with known order.

### 3. Handling support tickets

**Best fit:** Agentic workflow
Reason: stable outer scaffold with some dynamic routing and confidence-based escalation.

### 4. Travel planning with changing constraints

**Best fit:** Fully-agentic
Reason: open-ended search, comparison, retries, and adaptive planning.

### 5. Resume tailoring

**Best fit:** Prompt chain
Reason: predictable repeated steps with little need for autonomy.

### 6. Investigating a failed cloud deployment

**Best fit:** Fully-agentic
Reason: the path is uncertain and depends heavily on findings at each step.

### 7. Expense receipt processing

**Best fit:** Agentic workflow
Reason: defined pipeline with some judgment and exception handling inside it.

### 8. Summarizing one legal clause

**Best fit:** Single-shot
Reason: one direct explanation is enough.

### 9. Research assistant for buying a laptop

**Best fit:** Fully-agentic
Reason: dynamic search, comparison, conflict resolution, and changing path length.

### 10. Weekly business report generation

**Best fit:** Prompt chain
Reason: recurring structured process where consistency matters more than autonomy.

---

## One-line teaching takeaway

**Do not start with maximum autonomy. Start with the simplest approach that can reliably solve the task.**


## Table of contents

1. **Foundations** — from single-shot prompts to agent-driven systems
2. **Core building blocks** — planning, tools, memory, reflection, control, stopping
3. **Persona & system-prompt design**
4. **Self-improvement at inference time** (conceptual intro)
5. **The five workflow patterns**
   - 5.1 Prompt Chaining — patient intake pipeline
   - 5.2 Routing — triage classifier
   - 5.3 Parallelization — prescription safety checks
   - 5.4 Evaluator–Optimizer — discharge-summary QA loop
   - 5.5 Orchestrator–Worker — multi-specialist consultation
6. **ReAct-style orchestration** — reason / act / observe
7. **Self-improvement with memory from failures**
8. **Validation & reliability**
9. **Pattern comparison**
10. **Production considerations**
11. **Exercises**

---

## MedAssist

We'll build pieces of a clinical decision-support product called **MedAssist**.
MedAssist sits between a patient-facing app and a clinician's EHR. Across the notebook we'll implement:

- an intake pipeline that turns a patient's free-text message into a structured clinical summary,
- a triage router that dispatches messages to the right handler,
- a prescription safety engine that runs multiple checks in parallel,
- a discharge-summary writer with an automated QA loop,
- a complex-case consultation orchestrator that "calls in" specialist sub-agents,
- a small ReAct agent that can look up patient data, drugs, and guidelines.

At the end, every pattern reappears in a comparison table so you can see the tradeoffs side-by-side.

## Setup

Requirements: `openai>=1.60`, `pydantic>=2`. Set `OPENAI_API_KEY` in your environment.

Everywhere below we use `gpt-5-nano` with `reasoning.effort = "minimal"`. This
keeps the teaching demos fast and cheap; in production you'd tune these
per-call (small cheap model for classifiers, larger model for synthesizers).

In [24]:
import os, json
from dotenv import load_dotenv

import textwrap

import truststore
truststore.inject_into_ssl()

def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

        

load_dotenv('/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/openai_key.env')  # reads .env file in the current directory

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found! "
        "Make sure you have a .env file with: OPENAI_API_KEY=sk-..."
    )

pretty_print("API key loaded successfully.")

API key loaded successfully.


In [25]:
# !pip install -U openai pydantic

import os
import json
import time
import textwrap
from typing import List, Dict, Optional, Literal, Any
from pydantic import BaseModel, Field
from concurrent.futures import ThreadPoolExecutor, as_completed

from openai import OpenAI

client = OpenAI()

MODEL = "gpt-5-nano"
REASONING = {"effort": "minimal"}  # fast + cheap 
VERBOSITY = {"verbosity": "low"}
def llm_text(
    prompt,
    instructions=None,
    model=MODEL,
    reasoning=REASONING,
    tools=None,
    text = VERBOSITY
):
    """Simple text-in, text-out wrapper over the Responses API."""
    kwargs = dict(model=model, input=prompt, reasoning=reasoning)
    if instructions is not None:
        kwargs["instructions"] = instructions
    if tools is not None:
        kwargs["tools"] = tools
    resp = client.responses.create(**kwargs)
    return resp.output_text


def llm_parse(
    prompt,
    schema,
    instructions=None,
    model=MODEL,
    reasoning=REASONING,
):
    """Structured-output wrapper: returns a parsed Pydantic instance."""
    kwargs = dict(model=model, input=prompt, text_format=schema, reasoning=reasoning)
    if instructions is not None:
        kwargs["instructions"] = instructions
    resp = client.responses.parse(**kwargs)
    return resp.output_parsed


def pbox(label, content, width=96):
    """Pretty-print a labeled block for notebook readability."""
    bar = "─" * (width - 2)
    label_bar = f"── {label} " + bar[len(label) + 4:]
    print("\n┌" + label_bar)
    for line in str(content).splitlines() or [""]:
        if not line.strip():
            print("│")
            continue
        for wrapped in textwrap.wrap(line, width=width - 4) or [""]:
            print(f"│ {wrapped}")
    print("└" + bar)



### Synthetic medical data

Three small in-memory "databases" so the notebook runs without any external
service: a patient registry, a drug reference, and a mini clinical-guidelines store.
**All values are invented** and chosen to make the later examples pedagogically clean.

In [26]:
# -------------------------------------------------------------------------
# Synthetic data 
# -------------------------------------------------------------------------

PATIENTS: Dict[str, dict] = {
    "P001": {
        "id": "P001", "name": "Alex Chen", "age": 58, "sex": "M", "weight_kg": 82,
        "allergies": ["penicillin"],
        "conditions": ["type_2_diabetes", "hypertension", "ckd_stage_3"],
        "medications": ["metformin 1000mg BID", "lisinopril 20mg daily"],
        "recent_labs": {
            "hba1c_percent": 7.8, "egfr_ml_min": 52,
            "potassium_mmol_l": 4.6, "bp_mmhg": "142/88",
        },
    },
    "P002": {
        "id": "P002", "name": "Priya Sharma", "age": 34, "sex": "F", "weight_kg": 61,
        "allergies": [],
        "conditions": ["migraine"],
        "medications": ["sumatriptan 50mg PRN"],
        "recent_labs": {"hemoglobin_g_dl": 11.2},
        "pregnancy_status": "trying_to_conceive",
    },
    "P003": {
        "id": "P003", "name": "Rohan Patel", "age": 72, "sex": "M", "weight_kg": 74,
        "allergies": ["sulfa"],
        "conditions": ["atrial_fibrillation", "heart_failure_reduced_ef"],
        "medications": [
            "warfarin 5mg daily", "carvedilol 12.5mg BID", "furosemide 40mg daily",
        ],
        "recent_labs": {"inr": 2.6, "egfr_ml_min": 45, "potassium_mmol_l": 3.4},
    },
}

DRUGS: Dict[str, dict] = {
    "metformin": {
        "class": "biguanide",
        "contraindications": ["egfr_ml_min < 30"],
        "cautions": ["hold 48h around iodinated contrast"],
        "interactions": {},
    },
    "ibuprofen": {
        "class": "NSAID",
        "contraindications": ["ckd_stage_3", "heart_failure_reduced_ef", "active_gi_bleed"],
        "cautions": ["raises BP", "reduces renal perfusion"],
        "interactions": {
            "warfarin": "major — increases bleeding risk",
            "lisinopril": "reduces antihypertensive effect and renal perfusion",
        },
    },
    "amoxicillin": {
        "class": "penicillin",
        "contraindications": ["penicillin_allergy"],
        "cautions": [],
        "interactions": {"warfarin": "moderate — may elevate INR"},
    },
    "sumatriptan": {
        "class": "triptan",
        "contraindications": ["coronary_artery_disease", "uncontrolled_hypertension"],
        "cautions": ["avoid in pregnancy unless benefit outweighs risk"],
        "interactions": {"ssri": "moderate — serotonin syndrome risk"},
    },
    "ciprofloxacin": {
        "class": "fluoroquinolone",
        "contraindications": [],
        "cautions": ["tendon rupture risk in elderly", "QT prolongation"],
        "interactions": {
            "warfarin": "major — elevates INR",
            "tizanidine": "contraindicated",
        },
    },
    "nitrofurantoin": {
        "class": "urinary_antiseptic",
        "contraindications": ["egfr_ml_min < 30"],
        "cautions": ["avoid at term in pregnancy"],
        "interactions": {},
    },
}

GUIDELINES: Dict[str, str] = {
    "chest_pain_triage": (
        "Adult chest pain is an emergency until proven otherwise when accompanied by "
        "diaphoresis, dyspnea, radiation to arm/jaw, or hemodynamic instability. "
        "Send to emergency evaluation; obtain ECG within 10 minutes."
    ),
    "ckd_nsaids": (
        "Avoid NSAIDs in CKD stage 3 or worse (eGFR < 60). They reduce renal perfusion "
        "and can precipitate acute kidney injury, especially in combination with "
        "ACE inhibitors or ARBs."
    ),
    "warfarin_antibiotics": (
        "Many antibiotics (notably fluoroquinolones, macrolides, TMP-SMX, metronidazole) "
        "significantly elevate INR in warfarin patients. Recheck INR within 3–5 days of "
        "starting, and consider a temporary warfarin dose reduction."
    ),
    "migraine_pregnancy": (
        "In patients trying to conceive or pregnant, first-line acute migraine therapy is "
        "acetaminophen. Triptans are generally avoided unless benefits clearly outweigh "
        "risks; NSAIDs are avoided in the third trimester."
    ),
}


def lookup_patient(patient_id):
    return PATIENTS.get(patient_id)

def lookup_drug(drug_name):
    return DRUGS.get(drug_name.lower())

def lookup_guideline(topic):
    return GUIDELINES.get(topic)

print(f"Loaded {len(PATIENTS)} patients, {len(DRUGS)} drugs, {len(GUIDELINES)} guidelines.")

Loaded 3 patients, 6 drugs, 4 guidelines.


## 1. Foundations — from single-shot prompts to agents

A useful mental model is a **spectrum of delegation** from you (the developer) to the model:

```
┌─────────────────┬──────────────────┬──────────────────┬──────────────────┐
│  Single-shot    │   Prompt chain   │  Agentic workflow│  Fully-agentic   │
├─────────────────┼──────────────────┼──────────────────┼──────────────────┤
│ One prompt,     │ Fixed, hand-coded│ Structured flow  │ Model decides    │
│ one answer.     │ sequence of      │ with model-driven│ control flow,    │
│                 │ prompts.         │ decisions inside │ tools, when to   │
│                 │                  │ pre-built        │ stop — end-to-   │
│                 │                  │ scaffolding.     │ end.             │
└─────────────────┴──────────────────┴──────────────────┴──────────────────┘
  ← more developer control                              more model autonomy →
  lower cost/latency                                    higher cost/latency
  more predictable                                      more adaptive
```

The reason to move rightward is **not** that "agents are cool". It's that some
tasks genuinely require iteration, tool use, or dynamic decomposition to be
reliable. You are explicitly trading cost and latency for reliability and scope.

**Rules of thumb:**

- If a single well-designed prompt gets you there, **stop there**.
- If you can express the problem as a fixed pipeline, a **chain** is usually
  cheaper and more inspectable than an agent.
- Reach for **agentic patterns** when the problem space is too big or too
  branchy to hard-code — and accept the cost.

The small motivating demo below contrasts a single-shot answer against a
two-step chain for the same clinical question.

In [11]:
case = "58M with T2DM and CKD stage 3 (eGFR 52), on lisinopril. Wants ibuprofen for knee pain."

# --- Single-shot: just ask ---
single = llm_text(
    f"Clinical case: {case}\nIs ibuprofen safe here? Answer in 2 sentences.",
    instructions="You are a pharmacology assistant. Always flag safety concerns.",
)
pbox("Single-shot", single)

# --- Two-step chain: extract risks first, then reason ---
risks = llm_text(
    f"Case: {case}\nList, as a JSON array, every clinical risk factor "
    "that could contraindicate NSAIDs. Output JSON only.",
    instructions="Be exhaustive; cover renal, cardiac, GI, drug-interaction categories.",
)
reasoning_step = llm_text(
    f"Case: {case}\nKnown risk factors: {risks}\n\n"
    "Now decide: is ibuprofen safe? Answer in 2 sentences with a specific "
    "safer alternative if not.",
    instructions="You are a pharmacology assistant.",
)
pbox("Chain — risks (step 1)", risks)
pbox("Chain — decision (step 2)", reasoning_step)


┌── Single-shot ───────────────────────────────────────────────────────────────────────────────
│ Ibuprofen should be used with caution in CKD stage 3 and in patients on ACE inhibitors like
│ lisinopril because NSAIDs can worsen renal function and reduce kidney perfusion, potentially
│ leading to acute kidney injury. If NSAID is needed, use the lowest effective dose for the
│ shortest duration, consider alternatives (e.g., acetaminophen or topical agents), and
│ monitor kidney function and blood pressure closely; avoid concurrent use of NSAIDs with
│ lisinopril if possible. Safety flag: risk of nephrotoxicity and potential ACE
│ inhibitor–NSAID interaction.
└──────────────────────────────────────────────────────────────────────────────────────────────

┌── Chain — risks (step 1) ────────────────────────────────────────────────────────────────────
│ [
│   {"risk_factor": "Renal impairment (CKD stage 3; eGFR ~52 mL/min)"},
│   {"risk_factor": "Diabetes mellitus"},
│   {"risk_factor": "A

## 2. Core building blocks

Every serious agentic system is assembled from the same small set of pieces:

| Block | What it does | Example in MedAssist |
|---|---|---|
| **Planning** | Decides the next step(s) | *"For this case I need to check allergies, then interactions, then renal dosing."* |
| **Tools / function calling** | Reaches outside the weights | `lookup_patient`, `lookup_drug`, `lookup_guideline` |
| **Short-term memory** | Rolling context of the current run | Prior turns, intermediate results |
| **Long-term memory** | Persistent notes across runs | *"Last time I missed that eGFR<60 blocks NSAIDs — remember that."* |
| **Reflection / self-critique** | Generator inspects its own output | *"Did I actually cite the guideline?"* |
| **Control logic** | Routes and branches | `if urgency == "immediate": emergency_path()` |
| **Stopping conditions** | Keeps the loop bounded | *"Retry up to 3 times or stop when evaluator passes."* |

**A useful design test** before writing any code: *label each of these for your system*.
If you can't point at the "stopping condition" or the "planner", you don't yet have an
agentic design — you have an uncontrolled while-loop wrapped around an LLM.

## 3. Persona & system-prompt design

An agent's *persona* — the system prompt / `instructions` field in the Responses API —
is not decoration. It encodes:

- **Role** — who the model is pretending to be (*"a cautious internal medicine attending"*)
- **Norms** — what it must do (*"always list contraindications before recommending a drug"*)
- **Scope** — what it must refuse or defer (*"never give a definitive diagnosis in text"*)
- **Output contract** — shape, tone, length

The same underlying query produces *materially* different answers under different
personas. This is one of the cheapest, highest-leverage knobs you have.

In [12]:
q = (
    "A patient with hypertension asks if they can start an over-the-counter NSAID "
    "for back pain."
)

default_bot = llm_text(q, instructions="You are a helpful health bot.")

clinician = llm_text(
    q,
    instructions=(
        "You are a cautious internal medicine attending physician. "
        "Always (1) list relevant safety concerns first, (2) explain the mechanism, "
        "(3) suggest safer alternatives, (4) end with a when-to-see-a-doctor criterion. "
        "Keep answers under 120 words. Do not give a definitive diagnosis."
    ),
)

pbox("Default persona", default_bot)
pbox("Clinician persona", clinician)


┌── Default persona ───────────────────────────────────────────────────────────────────────────
│ Short answer: NSAIDs can raise blood pressure and may affect kidney function, so they should
│ be used with caution in hypertension. They’re not strictly contraindicated, but there are
│ important safety considerations.
│
│ Key points to discuss with the patient:
│ - Possible effects on blood pressure: NSAIDs (e.g., ibuprofen, naproxen) can cause fluid
│ retention and may reduce the effectiveness of antihypertensive medications, potentially
│ increasing BP.
│ - Kidney function: NSAIDs can impair kidney function, especially in those with hypertension,
│ diabetes, or concurrent kidney disease.
│ - Cardiovascular risk: In some patients, NSAIDs can be associated with increased risk of
│ heart attack or stroke, particularly with long-term use or higher doses.
│ - Dosing and duration: If an NSAID is deemed appropriate, use the lowest effective dose for
│ the shortest possible duration. Avoid ch

## 4. Self-improvement at inference time

"Self-improvement" here does **not** mean fine-tuning or weight updates. It means
using the model's own outputs to feed a better next generation, all at runtime. Four flavors matter:

1. **Self-reflection** — the model critiques its own draft.
2. **Self-correction** — it edits the draft based on that critique.
3. **Iterative refinement** — loop (1) → (2) until a stopping condition.
4. **Memory-based improvement** — failures are stored and consulted on future runs.

Two non-obvious points:

- **Critique and repair are distinct skills.** A model that can *spot* a bug in a
  diagnosis will not necessarily *fix* it cleanly. It is often worth splitting those
  prompts (and sometimes splitting the role into two "characters").
- **Self-critique without external grounding has a ceiling.** If the model is wrong
  about guideline *X*, it will confidently re-confirm itself. Pair self-critique with
  *execution-based* or *retrieval-based* feedback whenever possible.

The **Evaluator–Optimizer** pattern in §5.4 is the canonical runtime implementation
of (1)–(3). We'll handle (4) — memory — in §7.

## 5. The five workflow patterns

The rest of the notebook is built around five composable patterns. For each you get:

- the **intent** (one sentence)
- **when to reach for it**
- **when NOT to**
- a runnable MedAssist implementation
- the classic **failure modes**

| # | Pattern | Control flow | Core idea |
|---|---|---|---|
| 5.1 | Prompt Chaining | Sequential, fixed | Decompose into steps, each doing one thing well |
| 5.2 | Routing | Branching, fixed | Classify then dispatch to the right specialist prompt |
| 5.3 | Parallelization | Parallel, fixed | Fan out independent subtasks, aggregate |
| 5.4 | Evaluator–Optimizer | Iterative, bounded | Generator ↔ evaluator loop until quality gate passes |
| 5.5 | Orchestrator–Worker | Dynamic decomposition | Runtime plan → delegate to workers → synthesize |

### 5.1 Prompt Chaining — patient intake pipeline

```
┌────────┐   ┌────────┐   ┌────────┐   ┌────────┐   ┌────────┐
│ Extract│──▶│  Gate  │──▶│  DDx   │──▶│  Plan  │──▶│Translate│
└────────┘   └────┬───┘   └────────┘   └────────┘   └────────┘
                  │ emergency
                  ▼
            ┌──────────┐
            │ ER guide │   (short-circuit)
            └──────────┘
```

**Intent.** Break a hard task into a fixed sequence of smaller prompts, each step's
output feeding the next. Add a validation *gate* between steps when the cost of
propagating an error downstream is high.

**Reach for it when:** the task decomposes naturally, you can hand-specify the
steps, and each step benefits from being cheap, fast, and inspectable on its own.

**Don't use it when:** the decomposition depends on the *input* (you want an
orchestrator instead), or when the steps are actually independent (use
parallelization instead).

**Failure mode:** **error propagation**. A small hallucination in step 1 becomes a
"fact" by step 3. Mitigation: a validation step between stages that can
short-circuit the chain, and *selective context passing* — only forward the
structured fields the next step needs, not the whole transcript.

**MedAssist pipeline.** A patient types a free-text message. We chain:
1. **Extract** structured symptom summary (JSON)
2. **Gate** — is this safe to continue, or is it an emergency?
3. **Differential diagnosis** — ranked list with rationale
4. **Plan** — suggested next steps
5. **Translate** — patient-friendly summary

If the gate detects red-flag symptoms, we short-circuit straight to emergency guidance.

In [13]:
class IntakeExtract(BaseModel):
    chief_complaint: str
    duration: str
    associated_symptoms: List[str]
    red_flags: List[str] = Field(
        description="E.g. chest pain on exertion, syncope, thunderclap headache, focal weakness, sepsis signs."
    )
    relevant_history_mentioned: List[str]


class GateDecision(BaseModel):
    is_emergency: bool
    reason: str


def step1_extract(msg):
    return llm_parse(
        f'Patient message:\n"""{msg}"""\nExtract a structured intake summary.',
        IntakeExtract,
        instructions="Be conservative about red flags — prefer to flag than miss.",
    )

def step2_gate(extract):
    return llm_parse(
        f"Intake:\n{extract.model_dump_json(indent=2)}\n"
        "Decide whether to short-circuit to emergency guidance.",
        GateDecision,
        instructions=(
            "Return is_emergency=true if red_flags contain ANY life-threatening item "
            "(e.g., chest pain with exertion/diaphoresis/radiation, sudden severe headache, "
            "focal neuro deficit, suspected sepsis, difficulty breathing at rest)."
        ),
    )

def step3_differential(extract):
    return llm_text(
        f"Intake:\n{extract.model_dump_json(indent=2)}\n"
        "List a ranked differential diagnosis (3–5 items), each with one-line rationale.",
        instructions=(
            "You are a primary-care clinician. Do not commit to one diagnosis; "
            "list possibilities with rationale."
        ),
    )

def step4_plan(extract, ddx):
    return llm_text(
        f"Intake:\n{extract.model_dump_json(indent=2)}\n\n"
        f"Differential:\n{ddx}\n\n"
        "Propose next steps: tests, timeframe, self-care, when to seek care.",
        instructions="Be specific and actionable. Keep to ~120 words.",
    )

def step5_patient_summary(plan):
    return llm_text(
        f"Rewrite this clinical plan in plain, reassuring language at a 6th-grade "
        f"reading level, ~120 words, no medical jargon:\n\n{plan}",
    )


def intake_chain(msg):
    # Step 1 — extract
    extract = step1_extract(msg)

    # Step 2 — gate. Selective context passing: we ONLY forward the structured
    # extract, not the raw patient message, to reduce error propagation.
    gate = step2_gate(extract)
    if gate.is_emergency:
        return {
            "short_circuited": True,
            "extract": extract.model_dump(),
            "gate": gate.model_dump(),
            "patient_message": (
                "Based on what you described, this could be an emergency. "
                "Please call emergency services or go to the nearest emergency room now."
            ),
        }

    # Steps 3–5 — continue chain
    ddx = step3_differential(extract)
    plan = step4_plan(extract, ddx)
    summary = step5_patient_summary(plan)

    return {
        "short_circuited": False,
        "extract": extract.model_dump(),
        "differential": ddx,
        "plan": plan,
        "patient_message": summary,
    }


# --- Demo: one benign message, one red-flag message ---
benign = "I've had a sore throat and mild fever for 2 days, no trouble breathing."
red_flag = (
    "I have crushing chest pain radiating to my left arm and I'm sweating heavily. "
    "Started about 30 minutes ago."
)

print("═══ Benign case " + "═" * 60)
rb = intake_chain(benign)
pbox("Short-circuited?", rb["short_circuited"])
pbox("Extract", json.dumps(rb["extract"], indent=2))
pbox("Patient message", rb["patient_message"])

print("\n═══ Red-flag case " + "═" * 58)
rr = intake_chain(red_flag)
pbox("Short-circuited?", rr["short_circuited"])
pbox("Gate reason", rr["gate"]["reason"])
pbox("Patient message", rr["patient_message"])

═══ Benign case ════════════════════════════════════════════════════════════

┌── Short-circuited? ──────────────────────────────────────────────────────────────────────────
│ False
└──────────────────────────────────────────────────────────────────────────────────────────────

┌── Extract ───────────────────────────────────────────────────────────────────────────────────
│ {
│   "chief_complaint": "Sore throat and mild fever",
│   "duration": "2 days",
│   "associated_symptoms": [],
│   "red_flags": [
│     "fever",
│     "sore throat"
│   ],
│   "relevant_history_mentioned": []
│ }
└──────────────────────────────────────────────────────────────────────────────────────────────

┌── Patient message ───────────────────────────────────────────────────────────────────────────
│ Plan in plain language:
│
│ What we think now:
│ - This is most likely a viral sore throat (pharyngitis). If fever stays high, white spots,
│ or many sore-throat signs appear, we may check for strep throat.
│
│ Wha

### 5.2 Routing — triage classifier

```
                 ┌───────────┐
                 │  Router   │ (classifier)
                 └──┬──┬──┬──┘
          ┌─────────┘  │  └──────────┐
          ▼            ▼             ▼
    [emergency]   [medication]   [symptom]  …
```

**Intent.** Classify the incoming request into one of a few known categories,
then dispatch to the specialist prompt / model / tool best suited for it.

**Reach for it when:** you have a handful of clearly different sub-problems and
one generalist prompt is either too long or mediocre at all of them.

**Don't use it when:** there are really only two cases (just use an `if`), or
when the boundary is fuzzy and misclassification dominates your quality.
Fuzzy boundaries → evaluate with a calibrated confidence and use a *fallback branch*.

**Common variants:**
- **Task-based** — *billing vs clinical vs triage*
- **Complexity-based** — easy → small model, hard → larger model
- **Tool-based** — which tool is relevant (often implicit in function calling)
- **Model-based** — choose the model itself based on the input

**Failure mode:** misclassification. Always have a default/fallback branch, and
track confidence so you can escalate to a human when the classifier is unsure.

In [14]:
class TriageRoute(BaseModel):
    category: Literal[
        "emergency",           # life-threatening, bypass everything
        "medication_question", # prescription, side effects, interactions
        "symptom_triage",      # new/worsening symptoms, not emergent
        "lab_result_inquiry",  # questions about lab values
        "appointment_admin",   # scheduling, paperwork, billing
        "general_health_info", # general wellness questions
    ]
    urgency: Literal["immediate", "same_day", "within_week", "routine"]
    confidence: float = Field(ge=0.0, le=1.0)
    rationale: str


def triage_route(msg):
    return llm_parse(
        f'Patient message:\n"""{msg}"""',
        TriageRoute,
        instructions=(
            "You are a triage classifier. Pick exactly one category. "
            "Err on the side of higher urgency when in doubt. "
            "`confidence` is your own calibrated certainty (0–1)."
        ),
    )


# --- Specialist handlers (each is itself a small LLM call with its own persona) ---

def handle_emergency(msg):
    return (
        "🚨 This appears to be a medical emergency. Call emergency services now. "
        "[dedicated emergency handler branch]"
    )

def handle_medication(msg):
    return llm_text(
        f"Patient asks: {msg}\nAnswer as a clinical pharmacist. Keep under 150 words.",
        instructions=(
            "You are a clinical pharmacist. Always list major interactions and "
            "when to stop the medication."
        ),
    )

def handle_symptom(msg):
    return llm_text(
        f"Patient message: {msg}\nGive a brief self-care plan and a clear "
        "'see a doctor if...' list.",
        instructions="You are a primary care nurse. No diagnoses; symptom guidance only.",
    )

def handle_lab(msg):
    return llm_text(
        f"Patient asks about a lab value: {msg}\nExplain in plain language and "
        "indicate when to discuss with their doctor.",
        instructions="You explain labs; you never interpret them definitively.",
    )

def handle_admin(msg):
    return "[admin branch] Routed to the scheduling/billing team."

def handle_general(msg):
    return llm_text(
        f"General health question: {msg}\nAnswer in 80–120 words, neutral and evidence-based.",
    )


HANDLERS = {
    "emergency": handle_emergency,
    "medication_question": handle_medication,
    "symptom_triage": handle_symptom,
    "lab_result_inquiry": handle_lab,
    "appointment_admin": handle_admin,
    "general_health_info": handle_general,
}


def triage_and_respond(msg, confidence_threshold=0.55):
    route = triage_route(msg)
    # Fallback: low-confidence routes go to the safest generic handler,
    # flagged for human review.
    if route.confidence < confidence_threshold:
        response = (
            handle_general(msg)
            + "\n\n(Low routing confidence — flagged for human review.)"
        )
    else:
        response = HANDLERS[route.category](msg)
    return {"route": route.model_dump(), "response": response}


# --- Demo ---
messages = [
    "I need to reschedule my appointment next Tuesday.",
    "My blood pressure reading this morning was 178/104. Should I be worried?",
    "Can I take ibuprofen with my warfarin?",
    "Is it better to drink coffee in the morning or evening?",
]
for m in messages:
    print("\n" + "═" * 96)
    pbox("Incoming", m)
    out = triage_and_respond(m)
    pbox("Route", json.dumps(out["route"], indent=2))
    pbox("Response", out["response"])


════════════════════════════════════════════════════════════════════════════════════════════════

┌── Incoming ──────────────────────────────────────────────────────────────────────────────────
│ I need to reschedule my appointment next Tuesday.
└──────────────────────────────────────────────────────────────────────────────────────────────

┌── Route ─────────────────────────────────────────────────────────────────────────────────────
│ {
│   "category": "appointment_admin",
│   "urgency": "routine",
│   "confidence": 0.78,
│   "rationale": "The user is requesting to reschedule an existing appointment, which falls
│ under appointment administration. No urgent warning signs present."
│ }
└──────────────────────────────────────────────────────────────────────────────────────────────

┌── Response ──────────────────────────────────────────────────────────────────────────────────
│ [admin branch] Routed to the scheduling/billing team.
└─────────────────────────────────────────────────────

### 5.3 Parallelization — prescription safety checks

```
            ┌──────────┐
            │  Input   │
            └────┬─────┘
         ┌──────┼──────┐───────────┐
         ▼      ▼      ▼           ▼
       [allergy][renal][interact][contra]    (run concurrently)
         └──────┴──────┴───────────┘
                   │
                   ▼
              ┌─────────┐
              │Aggregate│
              └─────────┘
```

**Intent.** Run multiple LLM calls at the same time when the subtasks are
**independent**, and aggregate the outputs.

Two subflavors:
- **Sectioning** — split a task into disjoint sections, each handled by a
  separate call (e.g., check allergies *and* interactions *and* dosing, in parallel).
- **Voting** — run the *same* task several times and aggregate (useful for
  reducing variance on fuzzy judgments).

**Reach for it when:** subtasks don't depend on each other and latency matters.

**Don't use it when:** subtasks share state (chain them instead), or when
variance isn't actually a problem (voting adds cost for nothing).

**Tradeoff:** you pay in tokens for latency. Voting especially amplifies cost — use
it where a mistake is expensive.

**MedAssist use case.** Before confirming a prescription we need four independent
checks. Running them sequentially is wasted latency; they're independent *given
the patient record + proposed drug*.

In [15]:
class SafetyCheck(BaseModel):
    check_name: str
    passes: bool
    severity: Literal["none", "info", "caution", "warning", "contraindicated"]
    message: str


CHECKS = [
    (
        "allergy_check",
        "You check for allergy conflicts between the proposed drug and the patient's "
        "allergy list. Consider cross-reactivity (e.g., penicillin → amoxicillin).",
    ),
    (
        "renal_dosing_check",
        "You check whether the proposed drug is safe and correctly dosed given the "
        "patient's eGFR. Flag drugs contraindicated below certain eGFR thresholds.",
    ),
    (
        "interaction_check",
        "You check for interactions between the proposed drug and the patient's "
        "current medications. Flag major/contraindicated interactions loudly.",
    ),
    (
        "condition_contraindication_check",
        "You check whether the proposed drug is contraindicated by the patient's "
        "underlying conditions (e.g., NSAIDs in HFrEF/CKD, triptans in uncontrolled HTN).",
    ),
]


def run_single_check(
    check_name, check_instructions, patient, drug
):
    prompt = (
        f"Patient record:\n{json.dumps(patient, indent=2)}\n\n"
        f"Proposed drug: {drug}\n\n"
        f"Run ONLY the '{check_name}' check and return a SafetyCheck object."
    )
    return llm_parse(prompt, SafetyCheck, instructions=check_instructions)


def run_safety_checks_parallel(patient, drug):
    results: List[SafetyCheck] = []
    with ThreadPoolExecutor(max_workers=len(CHECKS)) as pool:
        futures = {
            pool.submit(run_single_check, name, instr, patient, drug): name
            for name, instr in CHECKS
        }
        for fut in as_completed(futures):
            results.append(fut.result())
    return results


def aggregate_safety(checks):
    rank = {"none": 0, "info": 1, "caution": 2, "warning": 3, "contraindicated": 4}
    worst = max(checks, key=lambda c: rank[c.severity])
    if worst.severity == "contraindicated":
        verdict = "BLOCK"
    elif worst.severity in {"warning", "caution"}:
        verdict = "REVIEW"
    else:
        verdict = "OK"
    return {
        "overall_verdict": verdict,
        "worst_severity": worst.severity,
        "checks": [c.model_dump() for c in checks],
    }


# --- Demo: prescribe ibuprofen to a patient with CKD + on ACE inhibitor ---
patient = lookup_patient("P001")
drug = "ibuprofen"

t0 = time.time()
checks = run_safety_checks_parallel(patient, drug)
elapsed = time.time() - t0

report = aggregate_safety(checks)
pbox("Overall verdict", f"{report['overall_verdict']} (worst severity: {report['worst_severity']})")
print(f"\nParallel runtime: {elapsed:.1f}s for {len(checks)} independent checks\n")
for c in report["checks"]:
    print(f"  • {c['check_name']:35s} severity={c['severity']:<16s} passes={c['passes']}")
    print(f"      {c['message']}")


┌── Overall verdict ───────────────────────────────────────────────────────────────────────────
│ REVIEW (worst severity: warning)
└──────────────────────────────────────────────────────────────────────────────────────────────

Parallel runtime: 6.1s for 4 independent checks

  • allergy_check                       severity=warning          passes=False
      Proposed drug ibuprofen may pose safety concerns for patient with penicillin allergy due to potential cross-reactivity? No direct cross-reactivity between beta-lactam (penicillin) and NSAIDs is established. However, NSAIDs like ibuprofen can affect kidney function and blood pressure; patient has CKD stage 3 and hypertension. Consider NSAID contraindication in CKD with eGFR 52 ml/min and potassium 4.6; monitor renal function and blood pressure. If opioid or alternative analgesic is needed, consider acetaminophen or topical agents. If NSAID use is necessary, use lowest effective dose for shortest duration and monitor. Allergy confl

### 5.4 Evaluator–Optimizer — discharge-summary QA loop

```
┌──────────┐   draft   ┌──────────┐   feedback   ┌──────────┐
│Generator │──────────▶│Evaluator │─────────────▶│  Revise  │
└──────────┘           └────┬─────┘              └────┬─────┘
                            │passes?                  │
                            ▼                         │
                        ┌───────┐                     │
                        │ DONE  │◀────────────────────┘ (or stop at budget)
                        └───────┘
```

**Intent.** One prompt generates a draft, a **second, differently-instructed**
prompt evaluates it against criteria, and the draft is revised until the evaluator
passes or a retry budget is hit.

**Why a separate evaluator?**
1. **Framing effect.** A model asked *"are there any issues with this?"* right
   after *"write me X"* is biased toward defending its own answer. A fresh
   evaluator persona with different criteria is substantially more rigorous.
2. **Separation of concerns.** The generator can be tuned for fluency; the
   evaluator can be tuned for strict correctness.

**Key design choices:**
- **Structured evaluation criteria.** Don't ask *"is this good?"* — ask for
  per-dimension scores and an explicit pass/fail.
- **Bounded retry budget.** Always have a hard cap (3 is a common default).
  Return the best-so-far when exhausted.
- **Deterministic stopping logic.** The code decides to stop, not the model.

**Reach for it when:** you can articulate *what good looks like*, and quality
matters more than latency.

**Don't use it when:** the evaluator can't meaningfully outperform the generator
(e.g., same model, same prompt → you're just paying double).

**MedAssist use case.** Generating a discharge summary. Missing meds or unclear
follow-up is a real patient-safety issue, so a QA loop is worth the latency.

In [ ]:
class DischargeEvaluation(BaseModel):
    passes: bool
    completeness_score: int = Field(
        ge=0, le=10,
        description="Includes diagnosis, meds with doses, follow-up with timeframe, red-flag warning signs.",
    )
    clarity_score: int = Field(
        ge=0, le=10, description="Patient-understandable language, no medical jargon.",
    )
    safety_score: int = Field(
        ge=0, le=10, description="No unsafe or contradictory instructions.",
    )
    missing_elements: List[str]
    concrete_fixes: List[str]


GENERATOR_INSTRUCTIONS = (
    "You are a hospitalist writing a discharge summary for the patient to take home. "
    "Include: diagnosis in plain language, full medication list with dose and frequency, "
    "follow-up plan with timeframe, red-flag symptoms that mean 'come back immediately'. "
    "Target length: 180–250 words, 7th-grade reading level."
)

EVALUATOR_INSTRUCTIONS = (
    "You are a discharge-safety reviewer — NOT the author. You read discharge summaries "
    "critically for missing information, jargon, and safety risks. Your scoring is strict. "
    "Set passes=true ONLY if all three scores are >=8 AND missing_elements is empty."
)


def generate_discharge(case):
    return llm_text(
        f"Case:\n{json.dumps(case, indent=2)}\n\nWrite the discharge summary.",
        instructions=GENERATOR_INSTRUCTIONS,
    )

def evaluate_discharge(case, draft):
    prompt = (
        f"Case:\n{json.dumps(case, indent=2)}\n\n"
        f"Discharge summary to review:\n{draft}\n\n"
        "Score strictly and list concrete fixes."
    )
    return llm_parse(prompt, DischargeEvaluation, instructions=EVALUATOR_INSTRUCTIONS)

def revise_discharge(case, prev_draft, feedback):
    prompt = (
        f"Case:\n{json.dumps(case, indent=2)}\n\n"
        f"Previous draft:\n{prev_draft}\n\n"
        f"Reviewer feedback:\n"
        f"  Missing: {feedback.missing_elements}\n"
        f"  Fixes:   {feedback.concrete_fixes}\n\n"
        f"Rewrite the full discharge summary addressing every fix. "
        f"Do not remove correct content."
    )
    return llm_text(prompt, instructions=GENERATOR_INSTRUCTIONS)




def discharge_with_qa(case, max_iters=3):
    history = []
    draft = generate_discharge(case)
    for i in range(1, max_iters + 1):
        evaluation = evaluate_discharge(case, draft)
        history.append({"iter": i, "draft": draft, "evaluation": evaluation.model_dump()})
        if evaluation.passes:
            return {"status": "passed", "iters": i, "final": draft, "history": history}
        if i == max_iters:
            return {"status": "budget_exhausted", "iters": i, "final": draft, "history": history}
        draft = revise_discharge(case, draft, evaluation)
    return {"status": "error", "iters": max_iters, "final": draft, "history": history}


# --- Demo ---
case = {
    "patient": PATIENTS["P003"],
    "admission_diagnosis": "Decompensated heart failure with hypokalemia",
    "hospital_course": (
        "IV diuresis with improvement in congestion; potassium replacement; "
        "warfarin held for planned procedure then restarted at prior dose."
    ),
    "discharge_meds": [
        "warfarin 5mg daily",
        "carvedilol 12.5mg BID",
        "furosemide 40mg BID (increased from 40mg daily)",
        "potassium chloride 20 mEq daily",
    ],
    "followup": "Cardiology clinic in 1 week; INR check in 3 days",
}

result = discharge_with_qa(case, max_iters=3)
pbox("Status", f"{result['status']} after {result['iters']} iteration(s)")

for h in result["history"]:
    ev = h["evaluation"]
    print(
        f"\n── Iter {h['iter']} ─ completeness={ev['completeness_score']} "
        f"clarity={ev['clarity_score']} safety={ev['safety_score']} passes={ev['passes']}"
    )
    if ev["missing_elements"]:
        print(f"    Missing: {ev['missing_elements']}")
    if ev["concrete_fixes"]:
        print(f"    Fixes:   {ev['concrete_fixes']}")

pbox("Final discharge summary", result["final"])


┌── Status ────────────────────────────────────────────────────────────────────────────────────
│ budget_exhausted after 3 iteration(s)
└──────────────────────────────────────────────────────────────────────────────────────────────

── Iter 1 ─ completeness=6 clarity=6 safety=6 passes=False
    Missing: ['Explicit discharge meds dose/form and frequency after changes (furosemide dose justification and BID vs daily consistency)', 'Renal function update and safety with potassium replacement (current K 3.4, aims for 3.5-5.0; missing target range)', 'INR monitoring plan specifics (target range, warfarin restart timing relative to procedure)', 'Warfarin interactions and bridging plan if procedure pending', 'Potassium replacement duration and monitoring plan (labs frequency post-discharge)', 'Follow-up lab plans (BMP, potassium, INR timing)', 'Discharge vitals and weight trend since admission (actual weight at discharge)', 'Clear documentation of diuretic stewardship post-discharge (why incr

### 5.5 Orchestrator–Worker — multi-specialist consultation

```
                     ┌──────────────┐
                     │ Orchestrator │  (plans at runtime:
                     └──────┬───────┘   which specialists to call)
              ┌─────────────┼──────────────┐
              ▼             ▼              ▼
        ┌──────────┐  ┌──────────┐  ┌──────────┐
        │Cardiology│  │Nephrology│  │ Pharmacy │    (workers, parallel)
        └────┬─────┘  └────┬─────┘  └────┬─────┘
             └─────────────┼──────────────┘
                           ▼
                    ┌─────────────┐
                    │ Synthesizer │
                    └─────────────┘
```

**Intent.** A top-level LLM call plans — **at runtime** — *which* subtasks to
create and *which worker* (sub-prompt / sub-agent) to route each to. A final
synthesizer merges the worker outputs.

**How is this different from chaining?** In chaining, *you* (the developer)
hand-specified the steps. Here the model decomposes the problem based on *this
particular input*. That flexibility is powerful — and also the source of most of
its failure modes.

**Reach for it when:** the right decomposition varies by input. Some cases need
cardiology + nephrology; others need endocrinology + pharmacy.

**Don't use it when:** you already know the steps. Fixed pipelines are cheaper
and more debuggable.

**Failure modes:** orchestrator omits a necessary worker; synthesizer gives
undue weight to one worker; unbounded planning loops. Mitigate with (a) a *fixed
menu* of workers, (b) a cap on how many are invoked, (c) explicit synthesis
instructions that require flagging conflicts.

**MedAssist use case.** A complex admission with overlapping problems — the
right set of specialists to consult depends on the case.

In [27]:
SPECIALISTS = {
    "cardiology":         "Cardiologist — arrhythmia, heart failure, antithrombotic therapy.",
    "nephrology":         "Nephrologist — renal function, electrolytes, renal dose adjustment.",
    "endocrinology":      "Endocrinologist — glycemic, thyroid, adrenal issues.",
    "pharmacy":           "Clinical pharmacist — drug selection, dosing, interactions.",
    "infectious_disease": "ID specialist — antimicrobial choice, stewardship, resistance.",
}

SpecName = Literal["cardiology", "nephrology", "endocrinology", "pharmacy", "infectious_disease"]

class ConsultPlan(BaseModel):
    selected: List[SpecName]
    rationale: str


def multi_specialist_consult(case):
    case_str = json.dumps(case, indent=2)

    # 1. Orchestrator — dynamic decomposition: which specialists matter here?
    plan = llm_parse(
        f"Case:\n{case_str}\n\nAvailable: {list(SPECIALISTS.keys())}\n"
        "Pick 2–4 whose input could change management.",
        ConsultPlan,
        instructions="You are the admitting attending. Be selective.",
    )

    # 2. Workers — run selected specialists in parallel
    def consult(spec):
        return spec, llm_text(
            f"Case:\n{case_str}\n\nGive your top 2–3 recommendations "
            "and flag any conflicts with other services.",
            instructions=SPECIALISTS[spec],
        )

    with ThreadPoolExecutor(max_workers=max(1, len(plan.selected))) as pool:
        outputs = dict(pool.map(consult, plan.selected))

    # 3. Orchestrator — synthesize into one plan
    joined = "\n\n".join(f"### {s.upper()}\n{outputs[s]}" for s in plan.selected)
    synthesis = llm_text(
        f"Case:\n{case_str}\n\nSpecialist inputs:\n{joined}\n\n"
        "Write one consolidated plan. Call out conflicts and your resolution. "
        "End with the top 5 action items in priority order.",
        instructions="You are the attending synthesizing specialist input into one plan.",
    )
    return plan, outputs, synthesis


# --- Demo ---
case = {
    "patient": PATIENTS["P003"],
    "presenting_problem": (
        "72M with AFib and HFrEF admitted with worsening dyspnea; workup shows "
        "UTI with E. coli, Cr 1.4→2.1, K 3.3, INR 4.8. "
        "On warfarin, carvedilol, furosemide."
    ),
}

plan, outputs, synthesis = multi_specialist_consult(case)
pbox("Plan", f"selected: {plan.selected}\nrationale: {plan.rationale}")
for s, out in outputs.items():
    pbox(s.upper(), out)
pbox("Synthesis", synthesis)


┌── Plan ──────────────────────────────────────────────────────────────────────────────────────
│ selected: ['cardiology', 'nephrology', 'infectious_disease', 'pharmacy']
│ rationale: Active management likely impacted by: (1) Cardiology: AFib/HFrEF optimization,
│ warfarin management with infection and rising INR; diuresis/CHF status. (2) Nephrology: AKI
│ progression (Cr 1.4→2.1), eGFR 45, electrolyte management (K 3.3), antibiotic dosing in CKD.
│ (3) Infectious disease: UTI with E. coli requiring antibiotics and consideration of drug
│ interactions with warfarin. (4) Pharmacy: warfarin interaction checks, dosing adjustments,
│ IV fluids, diuretic adjustments, and anticoagulation bridging or reversal if needed.
└──────────────────────────────────────────────────────────────────────────────────────────────

┌── CARDIOLOGY ────────────────────────────────────────────────────────────────────────────────
│ Here are the top 2–3 recommendations with attention to potential conflicts with o

## 6. ReAct-style orchestration

**ReAct** = **Reason** + **Act**. The model alternates between *thinking out loud*
("I should look up this patient's eGFR") and *taking an action* via a tool call
("call `lookup_patient("P001")`"). The tool's *observation* is fed back as the next
input. The loop continues until the model decides it has enough to answer.

```
 ┌─────────┐   thought    ┌──────────┐   action     ┌───────┐
 │  USER   │─────────────▶│  AGENT   │─────────────▶│ TOOLS │
 └─────────┘              └──────────┘              └───┬───┘
                                 ▲                      │
                                 │   observation        │
                                 └──────────────────────┘
```

Two things make ReAct work in practice:

1. **Tool schemas** that are unambiguous and narrowly scoped. Bad schemas
   cause bad tool use; no amount of clever prompting fixes them.
2. **Bounded loops.** Always cap `max_iterations`. An agent with no stop
   condition is a runtime bug waiting to cost you a lot of money.

The Responses API supports ReAct natively via function calling. We give the
model our tools; it returns `function_call` items when it wants to use them; we
execute them and pass observations back as `function_call_output` items.

In [18]:
# -------------------------------------------------------------------------
# Tool schemas for the Responses API
# -------------------------------------------------------------------------

TOOLS = [
    {
        "type": "function",
        "name": "lookup_patient",
        "description": "Look up a patient record by ID. Returns demographics, conditions, meds, labs.",
        "parameters": {
            "type": "object",
            "properties": {"patient_id": {"type": "string"}},
            "required": ["patient_id"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "lookup_drug",
        "description": "Look up a drug's class, contraindications, cautions, and interactions.",
        "parameters": {
            "type": "object",
            "properties": {"drug_name": {"type": "string"}},
            "required": ["drug_name"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "lookup_guideline",
        "description": (
            "Look up a short clinical guideline by topic key. "
            "Known keys: chest_pain_triage, ckd_nsaids, warfarin_antibiotics, migraine_pregnancy."
        ),
        "parameters": {
            "type": "object",
            "properties": {"topic": {"type": "string"}},
            "required": ["topic"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]

# Tool implementations (side of the fence we control)
TOOL_IMPLS = {
    "lookup_patient": lambda args: lookup_patient(args["patient_id"]) or {"error": "not found"},
    "lookup_drug":    lambda args: lookup_drug(args["drug_name"]) or {"error": "not found"},
    "lookup_guideline": lambda args: (
        {"guideline": lookup_guideline(args["topic"])}
        if lookup_guideline(args["topic"]) else {"error": "no guideline for that topic"}
    ),
}


REACT_INSTRUCTIONS = (
    "You are a cautious clinical assistant. Use the provided tools to gather evidence "
    "before answering. Think step by step. When you have enough information, give a "
    "final recommendation that cites the guideline(s) you used. Never recommend a "
    "medication without explicitly checking contraindications and interactions first."
)


def react_agent(user_question, max_steps=6, verbose=True):
    # Responses-API "input" is a list of typed items we grow turn by turn
    input_items: List[dict] = [{"role": "user", "content": user_question}]

    for step in range(1, max_steps + 1):
        resp = client.responses.create(
            model=MODEL,
            input=input_items,
            instructions=REACT_INSTRUCTIONS,
            tools=TOOLS,
            reasoning=REASONING,
        )

        # Collect any tool calls the model issued
        tool_calls = [item for item in resp.output if item.type == "function_call"]

        if not tool_calls:
            # No tool calls → the model produced a final answer
            if verbose:
                print(f"\n── Step {step}: final answer ──")
            return resp.output_text

        if verbose:
            print(f"\n── Step {step}: {len(tool_calls)} tool call(s) ──")

        # Echo each function_call back into input, then append its output
        for call in tool_calls:
            input_items.append({
                "type": "function_call",
                "name": call.name,
                "arguments": call.arguments,
                "call_id": call.call_id,
            })

            try:
                args = json.loads(call.arguments)
            except Exception:
                args = {}

            impl = TOOL_IMPLS.get(call.name)
            try:
                result = impl(args) if impl else {"error": f"unknown tool {call.name}"}
            except Exception as e:
                result = {"error": str(e)}

            if verbose:
                short = json.dumps(result)[:160]
                print(f"  • {call.name}({args}) → {short}")

            input_items.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": json.dumps(result),
            })

    return "[agent exceeded step budget without producing a final answer]"


# --- Demo 1: CKD + NSAID question ---
q1 = "Patient P001 has knee pain and wants ibuprofen. Is that safe? Cite the relevant guideline."
print("QUESTION:", q1)
ans1 = react_agent(q1, max_steps=6)
pbox("Agent answer", ans1)

# --- Demo 2: migraine in someone trying to conceive ---
print("\n" + "═" * 96)
q2 = "Patient P002 has a migraine today. What should she take? Cite the guideline you used."
print("QUESTION:", q2)
ans2 = react_agent(q2, max_steps=6)
pbox("Agent answer", ans2)

QUESTION: Patient P001 has knee pain and wants ibuprofen. Is that safe? Cite the relevant guideline.

── Step 1: 1 tool call(s) ──
  • lookup_patient({'patient_id': 'P001'}) → {"id": "P001", "name": "Alex Chen", "age": 58, "sex": "M", "weight_kg": 82, "allergies": ["penicillin"], "conditions": ["type_2_diabetes", "hypertension", "ckd_

── Step 2: 1 tool call(s) ──
  • lookup_drug({'drug_name': 'ibuprofen'}) → {"class": "NSAID", "contraindications": ["ckd_stage_3", "heart_failure_reduced_ef", "active_gi_bleed"], "cautions": ["raises BP", "reduces renal perfusion"], "i

── Step 3: 1 tool call(s) ──
  • lookup_guideline({'topic': 'ckd_nsaids'}) → {"guideline": "Avoid NSAIDs in CKD stage 3 or worse (eGFR < 60). They reduce renal perfusion and can precipitate acute kidney injury, especially in combination 

── Step 4: final answer ──

┌── Agent answer ──────────────────────────────────────────────────────────────────────────────
│ Summary of safety check
│ - Patient: P001 (Alex Chen), 58-ye

## 7. Self-improvement with memory from failures

The Evaluator–Optimizer pattern is runtime self-improvement *inside* a single
task. The other kind is *across* tasks: a system that remembers what went wrong
last time and avoids it next time. Same intuition as a postmortem folder in an
engineering team.

Below we build a tiny "lessons learned" store and show how folding it into the
prompt changes the next run. In a real system you'd want retrieval (to only
surface the relevant lessons), write-access controls, and expiry — but the
pattern is the same.

In [19]:
# Append-only "lessons learned" store
FAILURE_MEMORY: List[dict] = []


def record_lesson(scenario, mistake, corrected_principle):
    FAILURE_MEMORY.append({
        "scenario": scenario,
        "mistake": mistake,
        "principle": corrected_principle,
    })


def format_lessons(lessons, max_items=5):
    if not lessons:
        return "(no prior lessons)"
    return "\n".join(
        f"- In a '{l['scenario']}' case, do NOT {l['mistake']}; instead, {l['principle']}."
        for l in lessons[:max_items]
    )


def prescribe_with_memory(patient, drug):
    lessons = format_lessons(FAILURE_MEMORY)
    prompt = (
        f"Patient:\n{json.dumps(patient, indent=2)}\n\n"
        f"Proposed drug: {drug}\n\n"
        f"Lessons learned from previous cases:\n{lessons}\n\n"
        "Decide whether to approve, modify, or block the prescription. Be brief."
    )
    return llm_text(prompt, instructions="You are a cautious prescriber.")


# --- Run 1: cold, no memory ---
p = PATIENTS["P001"]
print(">>> Run 1 (cold, no memory)")
r1 = prescribe_with_memory(p, "ibuprofen")
pbox("Response 1", r1)

# Suppose a reviewer spots that we missed the eGFR issue. We capture that
# as a persistent lesson:
record_lesson(
    scenario="NSAID request in diabetic or CKD patient",
    mistake=(
        "approve NSAIDs without explicitly checking eGFR and ACE-inhibitor / ARB co-therapy"
    ),
    corrected_principle=(
        "BLOCK NSAIDs when eGFR < 60 OR patient is on an ACE inhibitor / ARB, "
        "and offer topical diclofenac or acetaminophen as alternatives"
    ),
)

# --- Run 2: same question, now with the lesson in memory ---
print("\n>>> Run 2 (with 1 lesson in memory)")
r2 = prescribe_with_memory(p, "ibuprofen")
pbox("Response 2", r2)

>>> Run 1 (cold, no memory)

┌── Response 1 ────────────────────────────────────────────────────────────────────────────────
│ Decision: Modify (not approve as-is).
│
│ Rationale and plan:
│ - Ibuprofen (NSAID) is contraindicated or discouraged in CKD stage 3, hypertension, and
│ diabetes due to risk of worsening renal function and blood pressure/volume status.
│ - Current eGFR 52 mL/min (CKD stage 3) with lisinopril, BP 142/88; NSAIDs can reduce renal
│ perfusion and worsen BP/creatinine.
│ - Allergies: penicillin only; no NSAID allergy noted.
│ - Consider alternatives for pain/inflammation: acetaminophen up to 3–4 g/day (adjust if
│ liver disease risk), topical NSAIDs, or non-pharmacologic options. If NSAID use is
│ essential, use shortest duration, lowest effective dose, with caution and
│ nephrology/prescriber consultation.
│
│ Recommended action:
│ - Do not approve ibuprofen at this time.
│ - If analgesia needed, switch to acetaminophen with max daily dose appropriate for
│ comorb

## 8. Validation & reliability

A blind spot in first-draft agent designs is relying on a *single* layer of
validation. Real systems stack them:

| Layer | Catches | Cost |
|---|---|---|
| **Schema validation** (Pydantic / JSON schema) | malformed outputs | ~free |
| **Rule checks in code** ("never recommend a drug the patient is allergic to") | known hard constraints | cheap |
| **LLM-based self-critique** | soft quality issues, unclear writing, missing sections | moderate |
| **LLM-based evaluator with different persona/criteria** | subtle reasoning errors, policy violations | moderate |
| **Execution-based feedback** (run the SQL / execute the code / compute the result) | factual and logical errors | varies |
| **Human-in-the-loop** | everything else | expensive, but necessary for high-stakes |

**Syntactic vs logical correctness.** Schema validation tells you the output
*parses*. It does not tell you it's *right*. A drug-prescription JSON can be
perfectly well-formed and still kill the patient. Pair schema checks with rule
checks with content review.

**External > self.** Whenever possible, ground validation in something *outside*
the model — a rule, a database, a code execution. Self-critique alone has a
ceiling: if the model is wrong about guideline *X* in the generator role,
it'll be wrong about *X* in the evaluator role too, because it's the same
weights. The evaluator/optimizer trick works because of *framing*, not
because of independence — remember that.

**Not every output needs every layer.** Match validation depth to consequence:

- Admin reply → schema check is fine.
- Drug prescription → schema + rules + evaluator + (usually) human sign-off.

The patterns in this notebook naturally map onto these layers. Chaining gives
you a gate between steps. Routing lets you apply *different* validation depths
to different categories. Parallelization lets you run the rule checks and the
LLM check side-by-side. Evaluator–Optimizer is one layer of LLM-as-judge. ReAct
adds *external* grounding via tools — which is why tool-using agents can
escape the self-critique ceiling that pure chains cannot.

## 9. Pattern comparison

| Dimension | Chaining | Routing | Parallelization | Evaluator–Optimizer | Orchestrator–Worker | ReAct |
|---|---|---|---|---|---|---|
| Control flow | fixed sequential | fixed branch | fixed fan-out | bounded loop | **runtime-planned** | runtime loop |
| Decomposition | by developer | by developer | by developer | by developer | **by model** | by model |
| Execution | sequential | one path | parallel | sequential | usually parallel workers | sequential tool use |
| Iteration | no | no | no | **yes** | typically no | yes (tool loop) |
| Typical cost | low | low (one handler) | N × generation | ~2–4× generation | 1 orch + N workers + 1 synth | variable, bounded by max_steps |
| Sweet spot | known pipeline | known categories | independent subtasks | quality gate matters | variable decomposition | need external lookups |
| Biggest failure mode | error propagation | misclassification | spurious independence | same-model self-bias | unbounded/incomplete plan | infinite tool loop |


**These patterns compose.** A production MedAssist might:

1. **Route** the incoming message (5.2),
2. In the *symptom* branch, run a **chain** (5.1),
3. Whose last step uses an **evaluator loop** (5.4),
4. And whose draft was produced by consulting **specialists** (5.5),
5. Each of which internally uses tools in a **ReAct** loop (§6).

When you see a real system that "works like magic," you're usually looking at
a thoughtful composition of 3–5 of these patterns, not one big clever prompt.

| Pattern             | Best example                                               |
| ------------------- | ---------------------------------------------------------- |
| Prompt Chaining     | Lecture transcript -> notes -> quiz                        |
| Parallelization     | Summarize many review batches at once                      |
| Routing             | Send support ticket to billing vs tech vs refund           |
| Evaluator–Optimizer | Draft discharge summary -> review -> revise                |
| Orchestrator–Worker | Complex case -> consult multiple specialists -> synthesize |
| ReAct               | Search flights iteratively using tools                     |


## 10. Production considerations

Things that matter once the demo works:

**Observability.** Log every LLM call: model, input, output, token counts, latency,
tool calls, retries. Every pattern in this notebook is a multi-call flow — you
cannot debug what you cannot see. Tools like LangSmith, OpenLLMetry, or a simple
structured logger get you most of the way.

**Budgets.**
- Token budget per request (so a runaway agent doesn't burn $20 per call)
- Iteration budget (max evaluator loops, max ReAct steps)
- Wall-clock timeout on each individual call

**Idempotency and retries.** Any parallel fan-out should handle partial failures.
Use retries with exponential backoff on transient errors (5xx, timeouts).
Do NOT retry 4xx errors — your request is wrong, not flaky.

**Structured outputs everywhere you can.** If a step can be JSON, make it JSON.
Free-text between agents is a major source of fragility.

**Tool sandboxing.** Tools that write to real systems (send email, order labs,
prescribe) need a confirmation step, a dry-run mode in development, and
rate-limiting. Never give an agent direct write access to production data
without a human-in-the-loop gate.

**Evaluation.** Build an offline eval set *before* you ship. Run it on every
prompt or model change. Track regressions per pattern — not just end-to-end.

**Model choice per call.** Not every call needs the biggest model. Triage
classifier → small fast model. Synthesis of specialist outputs → larger model.
The Responses API makes this trivial — just set `model=` per call.

**Safety for regulated domains.** Medical / legal / financial:
- Human sign-off on any consequential action
- Conservative defaults (refuse when uncertain)
- Clear refusal paths built into personas
- Loud logging of anything near a policy boundary
- Never hide disclaimers in the output

## 11. Exercises

1. **Confidence fallback in the router.** Extend the router so that when
   `confidence < 0.7` the message goes to a "human review" queue instead of the
   generic handler. Measure how often each message triggers it on a small test set.

2. **Pregnancy/lactation safety check.** Add a fifth parallel check to §5.3 that
   uses `patient.get("pregnancy_status")`. Run the full pipeline on patient `P002`
   with `sumatriptan` and verify the verdict changes.

3. **Cost-aware Evaluator–Optimizer.** Track cumulative tokens across iterations
   and stop early if the marginal improvement in scores falls below a threshold
   (e.g., <1 point across all dimensions between iterations).

4. **Compose patterns.** Wrap the ReAct agent (§6) *inside* the
   orchestrator's worker role (§5.5) so each specialist can use tools. How does
   cost and latency change? How does answer quality change?

5. **Regression eval harness.** Pick 10 patient messages, hand-label the
   expected routing category, and compute classifier accuracy. Now change the
   routing prompt and re-run. You just built a prompt regression test.

6. **Failure-memory retrieval.** Grow `FAILURE_MEMORY` past the point where
   you can paste all of it into every prompt. Add a simple retrieval step
   (keyword match or embedding similarity) that only pulls the most relevant
   lessons into context.

7. **Model-based routing.** Route simple queries to `gpt-5-nano` and complex
   ones (e.g., orchestrator+synthesis, evaluator) to a larger model. Measure the
   cost/quality tradeoff on your eval set.

8. **Failure injection.** Deliberately break each pattern once — force a schema
   parse error, force a tool to return `{"error": ...}`, force the evaluator to
   never pass. Verify your system handles each failure gracefully. Most production
   incidents are one of these three.

---
